# Ignite-3B Session S03 - Cond C gen 0-2 (same-model RSI MAIN)

**Goal**: run Cond C outer loop for gens 0-2 (8 candidates each, ~150 steps).

Setup:
1. Accelerator = GPU T4 x2
2. Internet ON
3. Persistence = Variables and Files
4. Kaggle Secret HF_TOKEN write access

Push v_0..v_2 to HF privado. Resume-safe via `--resume`.

In [ ]:
BASE_MODEL = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
HF_REPO = 'iterate-labs-ai/ignite-3b-cond-c'
BENCH = 'math'
BENCH_NAME = 'omni_math'
DATASET_TRAIN = 'data/ignite/omni_math_train.jsonl'
DATASET_DEV = 'data/ignite/omni_math_dev.jsonl'
DATASET_VAL = 'data/ignite/omni_math_val.jsonl'
OUT_ROOT = '/kaggle/working/cond_C_run'
GENS = 3   # this session: 0,1,2
CANDS = 8
STEPS = 150

In [ ]:
!pip install -q -U 'transformers>=4.46.0' 'peft>=0.13.0' 'datasets>=3.0.0' 'accelerate>=1.0.0' 'unsloth>=2025.1.0' 'trl>=0.12.0' 'vllm>=0.6.0' 'math-verify>=0.5.2' 'latex2sympy2' 'sympy' 'scipy' 'huggingface_hub'

In [ ]:
import os, subprocess
if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 's07-hybrid-agentic',
                    'https://github.com/iterate-labs-ai/caracal-1.git',
                    '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')
print(subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip())

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret('HF_TOKEN'))

In [ ]:
import subprocess, os, json, random
if not os.path.exists('data/ignite/omni_math.jsonl'):
    subprocess.run(['python', '-m', 'data.ignite.build_omni_math'], check=True)
rows = [json.loads(line) for line in open('data/ignite/omni_math.jsonl')]
random.Random(42).shuffle(rows)
n_train = int(0.7 * len(rows)); n_dev = int(0.15 * len(rows))
with open(DATASET_TRAIN, 'w') as f: [f.write(json.dumps(r) + '\n') for r in rows[:n_train]]
with open(DATASET_DEV, 'w') as f: [f.write(json.dumps(r) + '\n') for r in rows[n_train:n_train+n_dev]]
with open(DATASET_VAL, 'w') as f: [f.write(json.dumps(r) + '\n') for r in rows[n_train+n_dev:]]
print(f'train={n_train} dev={n_dev} val={len(rows)-n_train-n_dev}')

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-m', 'train.ignite.C_rsi_outer',
       '--base', BASE_MODEL,
       '--dataset-train', DATASET_TRAIN,
       '--dataset-dev', DATASET_DEV,
       '--dataset-val', DATASET_VAL,
       '--bench', BENCH,
       '--bench-name', BENCH_NAME,
       '--gens', str(GENS),
       '--cands', str(CANDS),
       '--steps', str(STEPS),
       '--out', OUT_ROOT,
       '--hf-repo', HF_REPO,
       '--resume']
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
log = [json.loads(line) for line in open(f'{OUT_ROOT}/log.jsonl')]
gens = [r for r in log if r.get('type') == 'generation']
print('accepted generations:')
for g in gens:
    print(f"  gen{g['gen']}: val_r={g['val_r']:.4f} delta={g['delta_pp']:.4f}")
print(f"\nHF repo: https://huggingface.co/{HF_REPO}")